In [0]:
%sql select * from sales_new_catalog.Silver.customers

In [0]:
dedup_df = spark.sql("select * from sales_new_catalog.Silver.customers")
dedup_df.show()

In [0]:
from pyspark.sql import functions as F

country_gold = (
    spark.read.table("sales_new_catalog.Silver.customers")
    .groupBy("country")
    .agg(F.count("customer_id").alias("TotalCustomers"))
    .orderBy(F.col("TotalCustomers").desc())
)

country_gold.show()

In [0]:
# 1. Create the city/country customer count aggregation
city_gold = (
    dedup_df.groupBy("city", "country")
    .agg(F.count("customer_id").alias("CustomerCount"))
    .orderBy(F.col("CustomerCount").desc())
)
city_gold.show()

In [0]:
# 2. Display the result in your notebook
city_gold.show(truncate=False)

In [0]:
# 3. Save the aggregated DataFrame as a CSV file to your Azure ADLS Gen2 container and catalog table
city_gold.write \
    .format("csv") \
    .option("header", "true") \
    .mode("overwrite") \
    .saveAsTable(
        "sales_new_catalog.Gold.region_summary_csv",
        path="abfss://workshop@sarmila01.dfs.core.windows.net/Gold/region_summary_csv"    
    )